# Straight Pipe CFD Validation

This notebook validates the OpenFOAM straight-pipe CFD case used as the baseline for the hemodynamic model comparison project. The geometry is intentionally simple: a straight circular pipe should produce Poiseuille-like laminar flow when driven at low Reynolds number.

The purpose here is not to compare reduced-order 0D/1D models yet. Instead, this notebook checks that the CFD solution, ParaView slicing/export workflow, and post-processing pipeline return physically consistent quantities before moving to tortuous vessels.

## 1. Introduction

A straight cylindrical pipe is the reference geometry for this workflow because it has a well-known analytical solution under steady, incompressible, laminar conditions. For a fully developed Newtonian flow, the axial velocity profile should be approximately parabolic, the pressure should decrease nearly linearly along the vessel, and the volumetric flow rate should be conserved between inlet, mid-vessel, and outlet sections.

The expected behavior is therefore Poiseuille-like:

- conserved flow rate along the pipe,
- linear pressure drop along the axial coordinate,
- centerline velocity close to twice the cross-sectional mean velocity,
- hydraulic resistance close to the analytical Poiseuille resistance.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

%matplotlib inline

pd.set_option('display.precision', 6)
pd.set_option('display.max_columns', 20)

plt.rcParams.update({
    'figure.figsize': (7.2, 4.4),
    'axes.grid': True,
    'grid.alpha': 0.28,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'legend.frameon': False,
})

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'scripts':
    REPO_ROOT = REPO_ROOT.parent

PIPE_DIR = REPO_ROOT / 'openfoam' / 'pipe'
POST_DIR = PIPE_DIR / 'postProcessing'
PARAVIEW_DIR = REPO_ROOT / 'data_parafoam'
OUTPUT_DIR = REPO_ROOT / 'output' / 'pipe_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PIPE_DIR, POST_DIR, PARAVIEW_DIR, OUTPUT_DIR

## 2. Load CFD Exported Data

The main quantitative source is the OpenFOAM post-processing directory for the completed straight-pipe case. The notebook also loads the available ParaView CSV exports from inlet, mid-vessel, and outlet slices so that the exported slicing workflow is visible in the validation record.

OpenFOAM incompressible pressure is stored as kinematic pressure. For hemodynamic reporting, pressure values are converted to pascals using `p_dynamic = rho * p_kinematic`.

In [ ]:
def read_surface_field_value(path: Path) -> dict:
    """Read an OpenFOAM surfaceFieldValue.dat file with scalar or vector area averages."""
    if not path.exists():
        raise FileNotFoundError(path)

    area = np.nan
    header = None
    value_line = None
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith('# Area'):
            area = float(line.split(':', 1)[1].strip())
        elif line.startswith('# Time'):
            header = line.lstrip('#').strip().split()
        elif not line.startswith('#'):
            value_line = line

    if value_line is None:
        raise ValueError(f'No data row found in {path}')

    time_text, rest = value_line.split(maxsplit=1)
    if '(' in rest and ')' in rest:
        values = [float(v) for v in re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', rest)]
        value = np.array(values, dtype=float)
    else:
        value = float(rest)

    return {'path': path, 'area_m2': area, 'time': float(time_text), 'value': value, 'header': header}


def read_xy_sample(path: Path) -> pd.DataFrame:
    """Read OpenFOAM sampled .xy files whose column names are stored in the comment header."""
    if not path.exists():
        raise FileNotFoundError(path)

    header = None
    for raw in path.read_text().splitlines():
        if raw.strip().startswith('#') and 'distance' in raw:
            header = raw.replace('#', ' ', 1).split()
            break
    if header is None:
        raise ValueError(f'Could not find column header in {path}')

    return pd.read_csv(path, comment='#', sep=r'\s+', names=header, engine='python')


def load_paraview_exports(directory: Path) -> dict:
    """Load ParaView CSV exports when they are present."""
    exports = {}
    for location in ['inlet', 'midslice', 'outlet']:
        slice_path = directory / f'{location}_integration.csv'
        integrated_path = directory / f'{location}_integration_variable.csv'
        exports[location] = {
            'slice': pd.read_csv(slice_path) if slice_path.exists() else pd.DataFrame(),
            'integrated': pd.read_csv(integrated_path) if integrated_path.exists() else pd.DataFrame(),
        }
    return exports


def parse_transport_nu(path: Path) -> float:
    text = path.read_text()
    match = re.search(r'\bnu\s+\[[^\]]+\]\s+([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)', text)
    if not match:
        raise ValueError(f'Could not parse nu from {path}')
    return float(match.group(1))


def geometry_from_block_mesh(path: Path) -> dict:
    text = path.read_text()
    radius_match = re.search(r'Radius\s+R\s+=\s+([0-9.eE+-]+)\s*m', text)
    length_match = re.search(r'Length\s+L\s+=\s+([0-9.eE+-]+)\s*m', text)
    if radius_match and length_match:
        radius = float(radius_match.group(1))
        length = float(length_match.group(1))
    else:
        vertices = re.findall(r'\(\s*([-+0-9.eE]+)\s+([-+0-9.eE]+)\s+([-+0-9.eE]+)\s*\)', text)
        xyz = np.array(vertices, dtype=float)
        radius = float(np.max(np.hypot(xyz[:, 0], xyz[:, 1])))
        length = float(np.max(xyz[:, 2]) - np.min(xyz[:, 2]))
    return {'radius_m': radius, 'diameter_m': 2 * radius, 'length_m': length, 'area_m2': np.pi * radius**2}


def pressure_to_pa(p_kinematic, rho):
    return np.asarray(p_kinematic, dtype=float) * rho

In [ ]:
# Physical parameters used by the OpenFOAM case.
rho = 1060.0  # kg/m^3, blood density used for dynamic pressure conversion
nu = parse_transport_nu(PIPE_DIR / 'constant' / 'transportProperties')
mu = rho * nu
geometry = geometry_from_block_mesh(PIPE_DIR / 'system' / 'blockMeshDict')

centreline = read_xy_sample(POST_DIR / 'sampleDict' / '144' / 'centreline.xy')
outlet_radial = read_xy_sample(POST_DIR / 'sampleDict' / '144' / 'outletRadial.xy')
paraview_exports = load_paraview_exports(PARAVIEW_DIR)

patch = {
    'inlet_U': read_surface_field_value(POST_DIR / 'patchAverage(patch=inlet,fields=(U))' / '144' / 'surfaceFieldValue.dat'),
    'outlet_U': read_surface_field_value(POST_DIR / 'patchAverage(patch=outlet,fields=(U))' / '144' / 'surfaceFieldValue.dat'),
    'inlet_p': read_surface_field_value(POST_DIR / 'patchAverage(patch=inlet,fields=(p))' / '144' / 'surfaceFieldValue.dat'),
    'outlet_p': read_surface_field_value(POST_DIR / 'patchAverage(patch=outlet,fields=(p))' / '144' / 'surfaceFieldValue.dat'),
}

case_summary = pd.DataFrame({
    'quantity': ['density rho', 'kinematic viscosity nu', 'dynamic viscosity mu', 'radius', 'diameter', 'length', 'analytical area'],
    'value': [rho, nu, mu, geometry['radius_m'], geometry['diameter_m'], geometry['length_m'], geometry['area_m2']],
    'units': ['kg/m^3', 'm^2/s', 'Pa s', 'm', 'm', 'm', 'm^2'],
})
display(case_summary)

loaded_summary = pd.DataFrame([
    {'dataset': 'centreline.xy', 'rows': len(centreline), 'columns': ', '.join(centreline.columns)},
    {'dataset': 'outletRadial.xy', 'rows': len(outlet_radial), 'columns': ', '.join(outlet_radial.columns)},
    *[
        {'dataset': f'ParaView {loc} slice CSV', 'rows': len(items['slice']), 'columns': ', '.join(items['slice'].columns)}
        for loc, items in paraview_exports.items()
    ],
    *[
        {'dataset': f'ParaView {loc} integrated CSV', 'rows': len(items['integrated']), 'columns': ', '.join(items['integrated'].columns)}
        for loc, items in paraview_exports.items()
    ],
])
display(loaded_summary)

## 3. Flow Rate Analysis

Flow conservation is the first baseline check. The inlet and outlet flow rates are computed from OpenFOAM area-averaged axial velocity and patch area. A mid-vessel estimate is included from the sampled axial velocity profile by using the Poiseuille relation `U_mean = U_max / 2`, which is appropriate for this straight-pipe validation case and is used only as a baseline consistency check.

In [ ]:
radius = geometry['radius_m']
diameter = geometry['diameter_m']
length = geometry['length_m']
area_patch = patch['outlet_U']['area_m2']
area_analytical = geometry['area_m2']

u_inlet = float(patch['inlet_U']['value'][2])
u_outlet = float(patch['outlet_U']['value'][2])
q_inlet = u_inlet * patch['inlet_U']['area_m2']
q_outlet = u_outlet * patch['outlet_U']['area_m2']

mid_row = centreline.iloc[(centreline['z'] - length / 2).abs().idxmin()]
u_mid_centerline = float(mid_row['U_z'])
u_mid_mean_poiseuille = u_mid_centerline / 2.0
q_mid = u_mid_mean_poiseuille * area_patch

flow_summary = pd.DataFrame({
    'location': ['inlet', 'mid-vessel', 'outlet'],
    'area_m2': [patch['inlet_U']['area_m2'], area_patch, patch['outlet_U']['area_m2']],
    'characteristic_U_m_per_s': [u_inlet, u_mid_mean_poiseuille, u_outlet],
    'Q_m3_per_s': [q_inlet, q_mid, q_outlet],
})
flow_summary['relative_to_outlet_percent'] = 100 * (flow_summary['Q_m3_per_s'] - q_outlet) / q_outlet

display(flow_summary)

q_spread_percent = 100 * (flow_summary['Q_m3_per_s'].max() - flow_summary['Q_m3_per_s'].min()) / flow_summary['Q_m3_per_s'].mean()
display(pd.DataFrame({
    'check': ['Q spread across inlet/mid/outlet'],
    'value_percent': [q_spread_percent],
    'interpretation': ['Small spread supports approximate conservation.'],
}))

In [ ]:
fig, ax = plt.subplots()
ax.bar(flow_summary['location'], flow_summary['Q_m3_per_s'], color=['#4477AA', '#66AA55', '#AA6644'])
ax.axhline(q_outlet, color='black', linewidth=1, linestyle='--', label='outlet reference')
ax.set_ylabel('Flow rate Q [m^3/s]')
ax.set_title('Flow Rate Conservation Check')
ax.legend()
plt.show()

## 4. Pressure Drop Analysis

The pressure drop is computed from area-averaged inlet and outlet pressures. The sampled centreline data are also used to inspect pressure evolution along the vessel. A nearly linear pressure-position curve is expected for fully developed laminar pipe flow.

In [ ]:
p_inlet_pa = float(pressure_to_pa(patch['inlet_p']['value'], rho))
p_outlet_pa = float(pressure_to_pa(patch['outlet_p']['value'], rho))
delta_p_patch = p_inlet_pa - p_outlet_pa

pressure_line = centreline.copy()
pressure_line['p_Pa'] = pressure_to_pa(pressure_line['p'], rho)
pressure_fit = np.polyfit(pressure_line['z'], pressure_line['p_Pa'], 1)
pressure_line['p_fit_Pa'] = np.polyval(pressure_fit, pressure_line['z'])
ss_res = np.sum((pressure_line['p_Pa'] - pressure_line['p_fit_Pa'])**2)
ss_tot = np.sum((pressure_line['p_Pa'] - pressure_line['p_Pa'].mean())**2)
pressure_r2 = 1 - ss_res / ss_tot
centreline_delta_p = pressure_line['p_Pa'].iloc[0] - pressure_line['p_Pa'].iloc[-1]

pressure_summary = pd.DataFrame({
    'quantity': ['P_inlet', 'P_outlet', 'DeltaP patch', 'DeltaP centreline', 'centreline dP/dz', 'linear fit R2'],
    'value': [p_inlet_pa, p_outlet_pa, delta_p_patch, centreline_delta_p, pressure_fit[0], pressure_r2],
    'units': ['Pa', 'Pa', 'Pa', 'Pa', 'Pa/m', '-'],
})
display(pressure_summary)

In [ ]:
fig, ax = plt.subplots()
ax.plot(pressure_line['z'], pressure_line['p_Pa'], color='#4477AA', linewidth=2, label='CFD centreline')
ax.plot(pressure_line['z'], pressure_line['p_fit_Pa'], color='#AA3377', linestyle='--', label='linear fit')
ax.scatter([0, length], [p_inlet_pa, p_outlet_pa], color='black', zorder=3, label='patch averages')
ax.set_xlabel('Axial position z [m]')
ax.set_ylabel('Pressure [Pa]')
ax.set_title('Pressure vs Axial Position')
ax.legend()
plt.show()

## 5. Velocity Analysis

For a laminar straight pipe, the fully developed velocity profile should be approximately parabolic. The sampled radial profile is compared with an ideal parabolic profile using the CFD mean flow rate as the reference mean velocity.

In [ ]:
profile = outlet_radial.copy()
profile['radius_signed_m'] = profile['x']
profile['radius_abs_m'] = np.abs(profile['x'])
profile['U_axial_m_per_s'] = profile['U_z']
profile['U_mag_m_per_s'] = np.sqrt(profile['U_x']**2 + profile['U_y']**2 + profile['U_z']**2)

u_mean_cfd = q_outlet / area_patch
u_max_profile = float(profile['U_axial_m_per_s'].max())
u_mean_profile_est = u_max_profile / 2.0
profile['U_poiseuille_m_per_s'] = 2 * u_mean_cfd * (1 - (profile['radius_abs_m'] / radius)**2)
profile['U_poiseuille_m_per_s'] = profile['U_poiseuille_m_per_s'].clip(lower=0)
profile_rmse = float(np.sqrt(np.mean((profile['U_axial_m_per_s'] - profile['U_poiseuille_m_per_s'])**2)))
profile_rmse_percent = 100 * profile_rmse / max(u_max_profile, np.finfo(float).eps)

velocity_summary = pd.DataFrame({
    'quantity': ['mean velocity from Q/A', 'max sampled axial velocity', 'profile-based mean estimate Umax/2', 'Umax / Umean', 'profile RMSE vs parabolic', 'profile RMSE / Umax'],
    'value': [u_mean_cfd, u_max_profile, u_mean_profile_est, u_max_profile / u_mean_cfd, profile_rmse, profile_rmse_percent],
    'units': ['m/s', 'm/s', 'm/s', '-', 'm/s', '%'],
})
display(velocity_summary)

display(profile[['radius_signed_m', 'U_axial_m_per_s', 'U_poiseuille_m_per_s', 'p']].head())

In [ ]:
fig, ax = plt.subplots()
ax.plot(profile['radius_signed_m'], profile['U_axial_m_per_s'], color='#4477AA', linewidth=2, label='CFD sampled profile')
ax.plot(profile['radius_signed_m'], profile['U_poiseuille_m_per_s'], color='#CC6677', linestyle='--', linewidth=2, label='Poiseuille reference')
ax.set_xlabel('Radial coordinate x [m]')
ax.set_ylabel('Axial velocity [m/s]')
ax.set_title('Outlet Velocity Profile')
ax.legend()
plt.show()

In [ ]:
velocity_plot_summary = velocity_summary[velocity_summary['quantity'].isin([
    'mean velocity from Q/A', 'max sampled axial velocity', 'profile-based mean estimate Umax/2'
])].copy()

fig, ax = plt.subplots()
ax.bar(velocity_plot_summary['quantity'], velocity_plot_summary['value'], color=['#4477AA', '#228833', '#CC6677'])
ax.set_ylabel('Velocity [m/s]')
ax.set_title('Velocity Statistics')
ax.tick_params(axis='x', rotation=25)
plt.show()

## 6. Reynolds Number

The Reynolds number is computed as

`Re = (rho * U * D) / mu`,

where `rho` is density, `mu` is dynamic viscosity, `U` is the characteristic mean velocity, and `D` is pipe diameter. For this validation case the characteristic velocity is the outlet mean velocity computed from `Q/A`.

In [ ]:
reynolds = rho * u_mean_cfd * diameter / mu
reynolds_alt = u_mean_cfd * diameter / nu

re_summary = pd.DataFrame({
    'quantity': ['rho', 'mu', 'nu', 'characteristic velocity U', 'diameter D', 'Re = rho U D / mu', 'Re = U D / nu'],
    'value': [rho, mu, nu, u_mean_cfd, diameter, reynolds, reynolds_alt],
    'units': ['kg/m^3', 'Pa s', 'm^2/s', 'm/s', 'm', '-', '-'],
})
display(re_summary)

display(Markdown(f'**Laminar assessment:** Re = {reynolds:.1f}, which is well below the usual pipe-flow transition range, so laminar Poiseuille-like behavior is expected.'))

## 7. Hydraulic Resistance

The CFD resistance is computed from the patch pressure drop and outlet flow rate:

`R_CFD = DeltaP / Q`.

The analytical Poiseuille resistance for a straight circular pipe is

`R = (8 * mu * L) / (pi * r^4)`.

In [ ]:
r_cfd = delta_p_patch / q_outlet
r_theory = 8 * mu * length / (np.pi * radius**4)
q_theory = delta_p_patch / r_theory
percent_difference = 100 * (r_cfd - r_theory) / r_theory

resistance_summary = pd.DataFrame({
    'quantity': ['R_CFD', 'R_Poiseuille', 'percentage difference', 'Q implied by Poiseuille at CFD DeltaP'],
    'value': [r_cfd, r_theory, percent_difference, q_theory],
    'units': ['Pa s / m^3', 'Pa s / m^3', '%', 'm^3/s'],
})
display(resistance_summary)

In [ ]:
resistance_plot = pd.DataFrame({
    'model': ['CFD', 'Poiseuille theory'],
    'resistance_Pa_s_per_m3': [r_cfd, r_theory],
})

fig, ax = plt.subplots()
ax.bar(resistance_plot['model'], resistance_plot['resistance_Pa_s_per_m3'], color=['#4477AA', '#AA6644'])
ax.set_ylabel('Resistance [Pa s / m^3]')
ax.set_title('CFD vs Theoretical Hydraulic Resistance')
for i, value in enumerate(resistance_plot['resistance_Pa_s_per_m3']):
    ax.text(i, value, f'{value:.2e}', ha='center', va='bottom')
plt.show()

## 8. Validation Discussion

The straight-pipe case behaves as expected for a laminar validation geometry when the following checks hold together:

- inlet, mid-vessel, and outlet flow rates are approximately conserved,
- pressure decreases nearly linearly along the pipe,
- the radial velocity profile is close to parabolic,
- Reynolds number remains in the laminar regime,
- CFD hydraulic resistance is close to the analytical Poiseuille resistance.

This case is important because it validates the CFD boundary conditions, exported slices, integrated variables, unit conversions, and analysis functions on a geometry with a known answer. Once this baseline is trustworthy, deviations observed in tortuous vessels are more likely to reflect geometric and modeling effects rather than post-processing errors.

In [ ]:
validation_checks = pd.DataFrame({
    'check': [
        'Flow conservation spread',
        'Pressure linearity R2',
        'Velocity profile RMSE / Umax',
        'Reynolds number',
        'Resistance difference',
    ],
    'value': [
        q_spread_percent,
        pressure_r2,
        profile_rmse_percent,
        reynolds,
        percent_difference,
    ],
    'units': ['%', '-', '%', '-', '%'],
    'baseline_interpretation': [
        'Low spread indicates near-conserved incompressible flow.',
        'Near 1.0 indicates linear pressure decay.',
        'Low value indicates parabolic profile agreement.',
        'Well below transition indicates laminar flow.',
        'Small difference indicates agreement with Poiseuille theory.',
    ],
})
display(validation_checks)

## 9. Figures

The core validation figures are rendered inline above:

- pressure vs axial position,
- flow-rate comparison,
- velocity statistics and radial velocity profile,
- CFD vs theoretical resistance comparison.

The notebook intentionally does not save every figure. The visible inline plots are the primary report artifacts.

## 10. Reusable Output Tables

The primary analysis remains visible in the notebook. A small set of final summary tables is also saved to `output/pipe_v2` because these values are useful later as the straight-pipe baseline for reduced-order and tortuous-vessel comparisons.

In [ ]:
summary_tables = {
    'straight_pipe_flow_summary.csv': flow_summary,
    'straight_pipe_pressure_summary.csv': pressure_summary,
    'straight_pipe_velocity_summary.csv': velocity_summary,
    'straight_pipe_reynolds_summary.csv': re_summary,
    'straight_pipe_resistance_summary.csv': resistance_summary,
    'straight_pipe_validation_checks.csv': validation_checks,
}

saved_files = []
for filename, table in summary_tables.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False)
    saved_files.append({'file': filename, 'path': str(path), 'rows': len(table)})

saved_summary = pd.DataFrame(saved_files)
display(saved_summary)

## Conclusion

The straight-pipe OpenFOAM case provides the baseline validation case for the hemodynamic model comparison workflow. The numerical checks above establish whether the CFD solution and exported post-processing quantities reproduce the expected laminar pipe-flow behavior before moving on to tortuous-vessel analysis.